# **Import**

In [6]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from scipy.stats import norm
from data_loader import Dataset
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score

# **Build From Scratch**


In [7]:
class CustomNB():

    def fit(self, X, y):

        self.X = X
        self.y = y
        n, m = self.X.shape

        self.is_cat = np.array([len(np.unique(self.X[:, j])) <= 20 for j in range(m)])
        X_cont = self.X[:, ~self.is_cat]

        self.classes = np.unique(self.y)
        self.r = len(self.classes)

        self.P_y = np.zeros(self.r)
        self.mean = np.zeros((self.r, X_cont.shape[1]))
        self.std = np.zeros((self.r, X_cont.shape[1]))

        for class_idx, class_value in enumerate(self.classes):

            X_class_cont = X_cont[y == class_value]

            self.mean[class_idx, :] = X_class_cont.mean(axis=0)
            self.std[class_idx, :] = X_class_cont.std(axis=0)

            self.P_y[class_idx] = np.sum(self.y == class_value) / n

    def predict(self, X):

        return np.array([self._predict(x) for x in X])

    def _predict(self, x):

        log_probs = []

        for class_idx in range(self.r):

            log_prob = np.log(self.P_y[class_idx])

            log_prob += np.sum(np.log(self._compute_P_x_y(class_idx, x)))

            log_probs.append(log_prob)

        return self.classes[np.argmax(log_probs)]

    def _compute_P_x_y(self, class_idx, x):

        x_cat = x[self.is_cat]
        x_cont = x[~self.is_cat]

        X_class = self.X[self.y == self.classes[class_idx]]
        X_class_cat = X_class[:, self.is_cat]
        n_class = X_class.shape[0]

        P_cat = []

        for j, value in enumerate(x_cat):

            freq = np.sum(X_class_cat[:, j] == value)
            P_cat.append(freq / n_class)

        mean = self.mean[class_idx]
        std = self.std[class_idx]
        P_cont = norm.pdf(x_cont, loc=mean, scale=std)

        return np.concatenate([P_cat, P_cont])

# **Load and Split**

In [8]:
dataset_m = Dataset("multiclass classification")
X_train_m, X_test_m, y_train_m, y_test_m = dataset_m.load_split_data()

dataset_b = Dataset("binary classification")
X_train_b, X_test_b, y_train_b, y_test_b = dataset_b.load_split_data()

# **Train, Test and Compare**

In [9]:
custom_model_m = CustomNB()
custom_model_m.fit(X_train_m, y_train_m)
y_pred_custom_m = custom_model_m.predict(X_test_m)
print(f"Custom Multiclass Classification Accuracy: {accuracy_score(y_test_m, y_pred_custom_m):.3f}")

custom_model_b = CustomNB()
custom_model_b.fit(X_train_b, y_train_b)
y_pred_custom_b = custom_model_b.predict(X_test_b)
print(f"Custom Binary Classification Accuracy: {accuracy_score(y_test_b, y_pred_custom_b):.3f}")

Custom Multiclass Classification Accuracy: 0.933
Custom Binary Classification Accuracy: 1.000


In [10]:
sklearn_model_m = GaussianNB()
sklearn_model_m.fit(X_train_m, y_train_m)
y_pred_sklearn_m = sklearn_model_m.predict(X_test_m)
print(f"Scikit-learn Multiclass Classification Accuracy: {accuracy_score(y_test_m, y_pred_sklearn_m):.3f}")

sklearn_model_b = GaussianNB()
sklearn_model_b.fit(X_train_b, y_train_b)
y_pred_sklearn_b = sklearn_model_b.predict(X_test_b)
print(f"Scikit-learn Binary Classification Accuracy: {accuracy_score(y_test_b, y_pred_sklearn_b):.3f}")

Scikit-learn Multiclass Classification Accuracy: 0.967
Scikit-learn Binary Classification Accuracy: 1.000
